# 🧠 Customer Segmentation Using RFM Analysis

## Overview

This project performs a comprehensive customer segmentation using **RFM Analysis** (Recency, Frequency, Monetary), a proven framework for identifying and categorizing customer behavior based on historical transaction data.

The dataset used originates from a UK-based online retail store, comprising over 1 million transactional records across multiple countries and two years of operations. The objective is to:

- Understand customer purchasing behavior
- Segment customers into actionable groups
- Support targeted marketing strategies and retention campaigns

---

## RFM Framework

RFM analysis segments customers using the following dimensions:

- **Recency (R):** How recently a customer made a purchase
- **Frequency (F):** How often a customer makes a purchase
- **Monetary (M):** How much money a customer spends

Each customer receives a score from 1 to 5 for each dimension, where higher scores indicate more desirable behavior (e.g., more recent, frequent, or valuable).

---

## Project Steps

1. 📥 **Data Loading and Initial Parsing**  
   Import the dataset, handle encoding and delimiters, and convert raw string data (e.g., price and date formats) into usable formats.

2. 🧾 **Dataset Overview and Initial Insights**  
   Generate summary statistics and data types. Identify null values, duplicated records, and extreme values such as negative quantities or prices.

3. 🔎 **Exploratory Data Analysis (EDA)**  
   Explore value distributions, data ranges, and customer behavior trends. Detect issues like data imbalance or unusual activity.

4. 🧼 **Data Cleaning for RFM Segmentation**  
   Remove irrelevant rows (e.g., missing Customer IDs, returns, and bad debt adjustments). Create a reliable `TotalPrice` metric for analysis.

5. 📊 **RFM Metrics Calculation**  
   Group transactions by customer to compute Recency, Frequency, and Monetary values. Define a snapshot date for consistent recency computation.

6. 🔢 **Calculate and Score RFM Metrics**  
   Score each RFM metric using quintiles (1–5), then construct composite `RFM_Segment` and `RFM_Score` indicators for ranking customers.

7. 🧩 **Segment Customers and Export RFM Results**  
   Assign business-relevant segment names (e.g., Champions, At Risk) based on RFM scores. Export the enriched dataset to CSV for further use.

8. 📊 **Interactive RFM Dashboard with Plotly**  
   Visualize segment distributions, spending behavior, and recency trends using dynamic bar charts, box plots, scatter plots, and pie charts.

---

## Tools & Technologies

- 🐍 **Python**: Data processing and analysis
- 📊 **Pandas, Matplotlib, Seaborn**: Data manipulation and visualization
- 🧮 **Quantile-based scoring**: Customer ranking methodology

---

## Business Application

RFM segmentation empowers marketing and product teams to:

- Target promotions to high-value customers
- Re-engage inactive segments with tailored offers
- Allocate resources to customer groups that drive revenue

This methodology is widely used in retail, e-commerce, and B2C businesses to drive customer-centric growth through data.

---


## 📥 Step 1: Data Loading and Initial Parsing

In this step, we import the dataset and ensure the date fields are correctly parsed to support time-based analysis.


---



In [1]:
import pandas as pd

# Load the CSV file with the correct delimiter and encoding
df = pd.read_csv('/mnt/data/online_retail_listing.csv', sep=';', encoding='ISO-8859-1')

# Convert 'InvoiceDate' to datetime format (day-first format based on your screenshot)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], dayfirst=True, errors='coerce')

# Preview structure and sample data
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1048575 non-null  object        
 1   StockCode    1048575 non-null  object        
 2   Description  1044203 non-null  object        
 3   Quantity     1048575 non-null  int64         
 4   InvoiceDate  1048575 non-null  datetime64[ns]
 5   Price        1048575 non-null  object        
 6   Customer ID  811893 non-null   float64       
 7   Country      1048575 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(5)
memory usage: 64.0+ MB


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,"6,95",13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,"6,75",13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,"6,75",13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,"2,1",13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,"1,25",13085.0,United Kingdom


## 🧾 Step 2: Dataset Overview and Initial Insights

After loading the dataset, we performed a structural analysis to understand its contents, format, and quality. Below are the key findings from `.info()` and `.head()`:

---

### 📦 Dataset Size and Structure

- **Total Records:** 1,048,575 transactions
- **Columns:** 8, including customer details, product data, pricing, and timestamps
- **Memory Footprint:** ~64 MB


---

### 📋 Column Breakdown

| Column        | Type         | Notes |
|---------------|--------------|-------|
| `Invoice`     | `object`     | Invoice numbers as strings (can include credit notes starting with "C") |
| `StockCode`   | `object`     | Product identifier codes |
| `Description` | `object`     | Product descriptions (some missing values) |
| `Quantity`    | `int64`      | Number of items purchased |
| `InvoiceDate` | `datetime64` | Timestamp of the transaction (already parsed to datetime) |
| `Price`       | `object`     | **Comma-separated string** at this stage (e.g., `6,95`) |
| `Customer ID` | `float64`    | Unique customer identifier; has **23% missing** values |
| `Country`     | `object`     | Country of the customer |

---

### 🔍 Key Observations

- **Missing Values:**  
  - `Customer ID` is missing in 236,000 records (~23% of total). These rows will be excluded from RFM analysis since customer-level grouping is essential.
  - `Description` is missing in ~4,300 rows, but this field is not needed for RFM and can be retained as-is or optionally cleaned.

- **Price Format:**  
  The `Price` field is still stored as a **string using commas as decimal separators** (e.g., `"6,95"`). This needs to be converted to a numeric format before any financial computation.

- **Initial Data Sample:**  
  The first few records show multiple purchases by a single customer (`Customer ID: 13085`) on **December 1, 2009**, with festive items like *Christmas lights* and *decorative boxes*. This supports the idea that purchases are highly seasonal—something to consider in segmentation.

---

### ✅ Next Step

We'll now clean the dataset by:
- Converting the `Price` field to numeric
- Dropping rows with missing customer data
- Removing returns or invalid transactions (negative quantities or prices)
- Creating a `TotalPrice` column for use in Monetary calculations


## 🔎 Step 3: Exploratory Data Analysis (EDA)

Before applying RFM segmentation, it's critical to explore the raw data to understand its structure, detect quality issues, and uncover potential inconsistencies. The following code performs a **comprehensive diagnostic review** of the dataset.


In [2]:
import pandas as pd

# Load dataset (update path if necessary)
df = pd.read_csv('/mnt/data/online_retail_listing.csv', sep=';', encoding='ISO-8859-1')

# Convert InvoiceDate to datetime format
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], dayfirst=True, errors='coerce')

# Replace commas in Price and convert to float
df['Price'] = df['Price'].str.replace(',', '.').astype(float)

# -------------------------------
# 1. Structural Overview
# -------------------------------

print("🔍 Dataset Info:")
df.info()

# -------------------------------
# 2. Missing Values
# -------------------------------

missing_values = df.isnull().sum().reset_index()
missing_values.columns = ['Column', 'MissingCount']
print("\n📉 Missing Values:\n", missing_values)

# -------------------------------
# 3. Summary Statistics
# -------------------------------

print("\n📊 Summary Statistics:\n", df.describe())

# -------------------------------
# 4. Unique Identifiers
# -------------------------------

print("\n🔢 Unique Entries:")
print("Unique Invoices:", df['Invoice'].nunique())
print("Unique Products:", df['StockCode'].nunique())
print("Unique Customers:", df['Customer ID'].nunique())
print("Unique Countries:", df['Country'].nunique())

# -------------------------------
# 5. Date Range
# -------------------------------

print("\n🗓️ Invoice Date Range:")
print("From:", df['InvoiceDate'].min(), "to", df['InvoiceDate'].max())

# -------------------------------
# 6. Invalid Entries (Negative Values)
# -------------------------------

neg_qty = df[df['Quantity'] < 0]
neg_price = df[df['Price'] < 0]

print("\n⚠️ Negative Quantity Rows:", len(neg_qty))
print("⚠️ Negative Price Rows:", len(neg_price))

# Optional: display first few of each to inspect
neg_qty.head(), neg_price.head()


🔍 Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1048575 non-null  object        
 1   StockCode    1048575 non-null  object        
 2   Description  1044203 non-null  object        
 3   Quantity     1048575 non-null  int64         
 4   InvoiceDate  1048575 non-null  datetime64[ns]
 5   Price        1048575 non-null  float64       
 6   Customer ID  811893 non-null   float64       
 7   Country      1048575 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 64.0+ MB

📉 Missing Values:
         Column  MissingCount
0      Invoice             0
1    StockCode             0
2  Description          4372
3     Quantity             0
4  InvoiceDate             0
5        Price             0
6  Customer ID        236682
7      Country             0

(     Invoice StockCode                    Description  Quantity  \
 178  C489449     22087       PAPER BUNTING WHITE LACE       -12   
 179  C489449    85206A   CREAM FELT EASTER EGG BASKET        -6   
 180  C489449     21895  POTTING SHED SOW 'N' GROW SET        -4   
 181  C489449     21896             POTTING SHED TWINE        -6   
 182  C489449     22083     PAPER CHAIN KIT RETRO SPOT       -12   
 
             InvoiceDate  Price  Customer ID    Country  
 178 2009-12-01 10:33:00   2.95      16321.0  Australia  
 179 2009-12-01 10:33:00   1.65      16321.0  Australia  
 180 2009-12-01 10:33:00   4.25      16321.0  Australia  
 181 2009-12-01 10:33:00   2.10      16321.0  Australia  
 182 2009-12-01 10:33:00   2.95      16321.0  Australia  ,
         Invoice StockCode      Description  Quantity         InvoiceDate  \
 179403  A506401         B  Adjust bad debt         1 2010-04-29 13:36:00   
 276274  A516228         B  Adjust bad debt         1 2010-07-19 11:24:00   
 403472  A


## 📊 Insights from Exploratory Analysis

After loading and parsing the dataset, we performed an initial examination of its structure, completeness, and statistical properties. This revealed several important findings that inform how the dataset should be cleaned and prepared for RFM analysis.

---

### 🔍 Dataset Structure

- **Rows:** 1,048,575
- **Columns:** 8
- **Memory Usage:** ~64 MB

This is a large and rich dataset containing transactional records across multiple years and customer segments.

---

### 📉 Missing Values

| Column       | Missing Values |
|--------------|----------------|
| `Customer ID` | 236,682 (~23%) |
| `Description` | 4,372 (~0.4%)  |

- The high proportion of missing `Customer ID` values is significant. Since RFM analysis is **customer-centric**, these rows will be excluded.
- Missing `Description` values are minimal and non-essential for our segmentation objective.

---

### 📊 Summary Statistics

- **Quantity**
  - Values range from **-74,215** to **74,215**
  - Standard deviation is large (`~133`), suggesting extreme outliers or returns.
- **Price**
  - Ranges from **-53,594.36** to **38,970.00**
  - Presence of large negative prices indicates **manual adjustments or debt corrections**.
- **Customer ID**
  - IDs range from 12346 to 18287, with 5,924 unique customers.

These statistics highlight the presence of **returns, cancellations, and possibly accounting anomalies** that must be filtered out before analysis.

---

### 🗓️ Time Coverage

- **Earliest Transaction:** December 1, 2009  
- **Latest Transaction:** December 4, 2011  
- **Coverage:** 2 full years of retail activity

This time range is more than sufficient for robust RFM scoring, particularly for calculating **Recency**.

---

### ⚠️ Data Quality Flags

| Issue             | Records Affected | Description |
|------------------|------------------|-------------|
| Negative Quantity | 22,697 rows      | Indicates **returns or cancellations** |
| Negative Price    | 5 rows           | Extreme values, labeled as **"Adjust bad debt"**, mostly with missing customer IDs |

These rows can distort frequency and monetary calculations and will be removed during data cleaning.

---

### ✅ Key Takeaways

- The dataset is **comprehensive and multi-dimensional**, with information suitable for customer-level analysis.
- **Data cleaning is essential** before segmentation. We will:
  - Drop rows with missing customer IDs
  - Remove transactions with negative quantity or price
  - Create new features like `TotalPrice` for use in Monetary scoring

This ensures the integrity and reliability of the RFM segmentation that follows.



## 🧼 Step 4: Data Cleaning for RFM Segmentation

Before computing RFM scores, it is essential to clean the dataset to ensure that only valid, complete, and relevant transactions are considered. The following code performs this cleaning process in a methodical way.

---

### 📥 Data Reload and Preprocessing

In [3]:
import pandas as pd

# Reload the file (adjust the path if needed)
df = pd.read_csv('/mnt/data/online_retail_listing.csv', sep=';', encoding='ISO-8859-1')

# Convert date and price fields
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], dayfirst=True, errors='coerce')
df['Price'] = df['Price'].str.replace(',', '.').astype(float)

# ---------------------------------------------
# Data Cleaning for RFM Segmentation
# ---------------------------------------------

# 1. Remove rows with missing Customer ID
df = df.dropna(subset=['Customer ID'])

# 2. Convert Customer ID to integer (standardization for groupby ops)
df['Customer ID'] = df['Customer ID'].astype(int)

# 3. Filter out rows with negative or zero Quantity (i.e., returns)
df = df[df['Quantity'] > 0]

# 4. Remove rows with zero or negative prices (usually bad debt adjustments)
df = df[df['Price'] > 0]

# 5. Create TotalPrice column used in Monetary calculation
df['TotalPrice'] = df['Quantity'] * df['Price']

# ✅ Data is now clean and ready for RFM segmentation
print(f"🧹 Cleaned dataset shape: {df.shape}")
df.head()


🧹 Cleaned dataset shape: (793309, 9)


<ipython-input-3-4f6021a18aec>:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Customer ID'] = df['Customer ID'].astype(int)
<ipython-input-3-4f6021a18aec>:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['TotalPrice'] = df['Quantity'] * df['Price']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


## ✅ Post-Cleaning Results & Dataset Validation

After applying the data cleaning pipeline, we validated the shape and structure of the dataset to ensure it is ready for RFM segmentation.

---

### 🧹 Cleaned Dataset Overview

```python
🧹 Cleaned dataset shape: (793,309, 9)


## 📊 Step 5: RFM Metrics Calculation

With the cleaned transactional dataset, the next step in our customer segmentation process is to compute the core **RFM metrics** — Recency, Frequency, and Monetary value — for each customer. These metrics summarize purchasing behavior and are foundational for classifying customers into

*   List item
*   List item

meaningful segments.


In [5]:
import pandas as pd
from datetime import timedelta

# Load dataset from Colab path or re-upload manually if needed
df = pd.read_csv('/mnt/data/online_retail_listing.csv', sep=';', encoding='ISO-8859-1')

# Convert data types
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], dayfirst=True, errors='coerce')
df['Price'] = df['Price'].str.replace(',', '.').astype(float)

# Clean data
df = df.dropna(subset=['Customer ID']).copy()
df.loc[:, 'Customer ID'] = df['Customer ID'].astype(int)
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]
df.loc[:, 'TotalPrice'] = df['Quantity'] * df['Price']

# Calculate RFM metrics
snapshot_date = df['InvoiceDate'].max() + timedelta(days=1)

rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'Invoice': 'nunique',                                     # Frequency
    'TotalPrice': 'sum'                                       # Monetary
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']
rfm.head()


,CustomerID,Recency,Frequency,Monetary
0,12346.0,321,12,77556.46
1,12347.0,35,7,5408.50
2,12348.0,71,5,2019.40
3,12349.0,14,4,4428.69
4,12350.0,305,1,334.40


## 📈 RFM Table Preview and Initial Visual Insights

After calculating the **Recency**, **Frequency**, and **Monetary** values for each customer, we obtained a structured RFM table containing one row per customer. Here's an overview of the key outputs and what they tell us:

---

### 🧾 RFM Table Sample

| CustomerID | Recency | Frequency | Monetary |
|------------|---------|-----------|----------|
| 12346      | 321     | 12        | 77,556   |
| 12347      | 35      | 7         | 540      |
| 12348      | 71      | 5         | 201      |
| 12349      | 14      | 4         | 4,428    |
| 12350      | 305     | 1         | 33       |

- **Recency**: Number of days since the customer's last purchase, counted from a snapshot date (Dec 5, 2011).
- **Frequency**: Number of unique purchase invoices.
- **Monetary**: Total amount spent by the customer.

> For example, customer `12346` is a **high spender** (₤77,556) but hasn't purchased in **321 days**, indicating a previously valuable but now **inactive customer**.

---

### 📊 Distribution Insights

The visualizations below the table provide basic exploratory patterns:

#### **Distributions**
- **Recency** is right-skewed: many customers haven't purchased recently.
- **Frequency** shows most customers purchased just a few times.
- **Monetary** has extreme outliers — a few customers spent significantly more than others.

#### **2D Scatter Plots**
- Show the relationships between:
  - Frequency vs. Recency
  - Monetary vs. Frequency
  - Recency vs. Monetary
- These plots help detect behavioral clusters and potential outliers.

#### **Time Series & Summary**
- The line charts and summary statistics further confirm that:
  - Spending behavior is uneven.
  - High monetary values are concentrated in a small segment of customers.

---

### ✅ Next Step: RFM Scoring and Segmentation

Now that the raw RFM values are ready, we’ll:
1. Score each metric on a 1–5 scale (e.g., using quintiles).
2. Concatenate RFM scores to create behavioral segments
3. Interpret and label key customer personas (e.g., Champions, At-Risk, Loyal).

This step transforms numeric features into **actionable marketing insights**.


## 🔢 Step 6: Calculate and Score RFM Metrics

In this step, we compute the **Recency**, **Frequency**, and **Monetary** values for each customer, and then assign scores from 1 to 5 based on customer behavior using quintile-based ranking.

---

In [6]:
import pandas as pd
from datetime import timedelta

# Load dataset
df = pd.read_csv('/mnt/data/online_retail_listing.csv', sep=';', encoding='ISO-8859-1')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], dayfirst=True, errors='coerce')
df['Price'] = df['Price'].str.replace(',', '.').astype(float)

# Clean dataset
df = df.dropna(subset=['Customer ID']).copy()
df.loc[:, 'Customer ID'] = df['Customer ID'].astype(int)
df = df[df['Quantity'] > 0].copy()
df = df[df['Price'] > 0].copy()
df.loc[:, 'TotalPrice'] = df['Quantity'] * df['Price']  # Safe assignment

# Calculate RFM metrics
snapshot_date = df['InvoiceDate'].max() + timedelta(days=1)
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'Invoice': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# Score RFM using quintiles
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

# Create RFM segment and overall score
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Score'] = rfm[['R_Score', 'F_Score', 'M_Score']].sum(axis=1)

# Preview results
rfm.head()


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Segment,RFM_Score
0,12346.0,321,12,77556.46,2,5,5,255,12
1,12347.0,35,7,5408.50,4,4,5,445,13
2,12348.0,71,5,2019.40,3,4,4,344,11
3,12349.0,14,4,4428.69,5,3,5,535,13
4,12350.0,305,1,334.40,2,1,2,212,5


## 🧮 RFM Scoring Results

After computing the Recency, Frequency, and Monetary values for each customer, we assigned RFM scores using **quintile-based segmentation** (1 to 5 scale). Each metric was scored independently and then combined to create interpretable customer segments.

---

### 📊 Sample of Scored RFM Table

| CustomerID | Recency | Frequency | Monetary | R_Score | F_Score | M_Score | RFM_Segment | RFM_Score |
|------------|---------|-----------|----------|---------|---------|---------|-------------|-----------|
| 12346      | 321     | 12        | 77,556.46 | 2       | 5       | 5       | 255         | 12        |
| 12347      | 35      | 7         | 5,408.50  | 4       | 4       | 5       | 445         | 13        |
| 12348      | 71      | 5         | 2,019.40  | 3       | 4       | 4       | 344         | 11        |
| 12349      | 14      | 4         | 4,428.69  | 5       | 3       | 5       | 535         | 13        |
| 12350      | 305     | 1         | 334.40    | 2       | 1       | 2       | 212         | 5         |

---

### 🧠 What These Scores Mean

- **R_Score (Recency)**: Lower values = more recent activity → higher score (5 = very recent purchase)
- **F_Score (Frequency)**: More invoices = higher score (5 = very frequent buyer)
- **M_Score (Monetary)**: More spending = higher score (5 = top spender)

Each customer gets a **3-digit segment label** (`RFM_Segment`) such as:
- `255`: Moderate recency, top frequency and monetary value
- `445`: Strong recency and spending, with good frequency
- `212`: Low scores in all metrics — a likely disengaged, low-value customer

The **RFM_Score** (sum of R, F, M scores) provides a simplified numeric ranking for quick sorting or clustering.

---

### ✅ Why This Step Matters

RFM scoring allows us to:
- **Segment customers** into actionable groups (e.g. Champions, At-Risk, Big Spenders)
- **Prioritize marketing and retention** strategies based on customer value
- Build targeted campaigns (e.g. win-back emails, loyalty rewards, reactivation offers)

This scoring system provides a **data-driven foundation** for customer relationship management and lifetime value optimization.

---

### 🔜 Next Step

We'll now define **RFM segments** such as:
- **Champions**: 555, 545, 554
- **Loyal Customers**: 4xx, 5xx with high Frequency
- **At-Risk**: Low R_Score but previously high M and F
- **Hibernating or Lost**: Low across all three scores

These labeled personas help translate data into **actionable business insights**.


## 🧩 Step 7: Segment Customers and Export RFM Results

In this step, we go beyond RFM scoring by assigning meaningful **business segments** to each customer based on their Recency, Frequency, and Monetary scores. Then, we export the results for reporting or integration into other systems.


---

## 🔢 RFM Scoring Logic and Segment Mapping

This section describes how we convert raw RFM metrics into meaningful scores and business segments using quantile-based binning and custom logic.

### RFM Scoring Using Quantiles

We score each of the **Recency**, **Frequency**, and **Monetary** metrics on a scale from 1 to 5. This scoring system is based on quintiles so that each customer is assigned a score relative to the overall distribution:

- **Recency (R_Score):**
  - **Lower recency is better** (i.e., customers who purchased recently are more valuable).
  - We use `pd.qcut` to split the `Recency` values into 5 quantiles.
  - The labels `[5, 4, 3, 2, 1]` mean:
    - A score of **5** indicates a very recent purchase (top 20% of recency, i.e., smallest recency values).
    - A score of **1** indicates a long time since the last purchase.

- **Frequency (F_Score):**
  - **Higher frequency is better** (i.e., customers with more purchases are more engaged).
  - We rank the `Frequency` column using `rank(method='first')` before splitting into quantiles.
  - The labels `[1, 2, 3, 4, 5]` assign a score where:
    - A score of **5** means the customer has made many purchases.
    - A score of **1** means relatively few purchases.

- **Monetary (M_Score):**
  - **Higher monetary value is better** (i.e., customers who spend more are more valuable).
  - Similarly, we use `pd.qcut` on the `Monetary` column to assign scores of 1 to 5.
  - A score of **5** signifies high spending, while **1** means low spending.



---

## 🏷️ Segment Naming and Assignment Logic

After scoring each customer based on their Recency, Frequency, and Monetary values, we assigned them to intuitive, business-relevant segments. These segment names are based on common customer behavior patterns observed in marketing and lifecycle analytics. Below is the logic behind each category name and its assignment criteria:

---

### 🔝 Champions  
**Criteria**: R_Score ≥ 4, F_Score ≥ 4, M_Score ≥ 4  
These are the best customers — they purchased recently, purchase often, and spend the most.  
**Rationale**: They represent the highest customer lifetime value.  
**Action**: Reward with loyalty programs, early access, exclusive offers.

---

### 🔁 Loyal Customers  
**Criteria**: R_Score ≥ 3, F_Score ≥ 4  
Customers who buy frequently and fairly recently, though they may not spend as much as Champions.  
**Rationale**: They are committed and consistent buyers.  
**Action**: Strengthen the relationship with upselling and engagement campaigns.

---

### 🌱 Potential Loyalists  
**Criteria**: R_Score ≥ 4, F_Score ≤ 2  
These are recent buyers with low frequency — new customers with strong potential to become loyal.  
**Rationale**: They’re engaged now and can be nurtured into long-term buyers.  
**Action**: Provide onboarding, personalized follow-ups, and incentives to return.

---

### ✨ New Customers  
**Criteria**: R_Score = 5  
Just made their first purchase very recently.  
**Rationale**: Their behavior is not yet established, but timing is ideal for activation.  
**Action**: Deliver onboarding flows, product education, and second-purchase incentives.

---

### ⚠️ At Risk  
**Criteria**: R_Score ≤ 2, F_Score ≥ 4  
These customers used to buy frequently but haven’t returned in a while.  
**Rationale**: They were once engaged and valuable, but are slipping away.  
**Action**: Win-back campaigns, special offers, or re-engagement emails.

---

### 💤 Hibernating  
**Criteria**: R_Score ≤ 2, F_Score ≤ 2, M_Score ≤ 2  
Low engagement across the board — haven’t purchased recently, rarely buy, and spend little.  
**Rationale**: Disengaged customers with low current value.  
**Action**: Send occasional reactivation messages or remove from active lists.

---

### ❓ Others  
**Criteria**: Do not meet the above patterns  
These customers exhibit mixed or undefined behavior.  
**Rationale**: They don’t clearly fit any segment and may require deeper profiling.  
**Action**: Monitor for future changes or explore with clustering techniques.

---

These segment labels make RFM insights practical and actionable, enabling marketing, sales, and CRM teams to tailor strategies based on customer value and engagement stage.


In [ ]:
import pandas as pd
from datetime import timedelta
from IPython.display import FileLink, display

# Load dataset
df = pd.read_csv('/mnt/data/online_retail_listing.csv', sep=';', encoding='ISO-8859-1')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], dayfirst=True, errors='coerce')
df['Price'] = df['Price'].str.replace(',', '.').astype(float)

# Clean data
df = df.dropna(subset=['Customer ID']).copy()
df.loc[:, 'Customer ID'] = df['Customer ID'].astype(int)
df = df[df['Quantity'] > 0].copy()
df = df[df['Price'] > 0].copy()
df.loc[:, 'TotalPrice'] = df['Quantity'] * df['Price']

# RFM calculation
snapshot_date = df['InvoiceDate'].max() + timedelta(days=1)
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'Invoice': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# RFM scoring
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Score'] = rfm[['R_Score', 'F_Score', 'M_Score']].sum(axis=1)

# Segment mapping
def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 4:
        return 'Loyal Customers'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'Potential Loyalists'
    elif row['R_Score'] == 5:
        return 'New Customers'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 4:
        return 'At Risk'
    elif row['R_Score'] <= 2 and row['F_Score'] <= 2 and row['M_Score'] <= 2:
        return 'Hibernating'
    else:
        return 'Others'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

# Count customers per segment
segment_counts = rfm['Segment'].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Customer Count']

# Display the segment count summary using Pandas
print("📊 Customer Segmentation Summary:")
display(segment_counts)

# Export to CSV and trigger download in Colab
csv_path = '/mnt/data/rfm_customer_segments.csv'
rfm.to_csv(csv_path, index=False)

# Trigger download
from google.colab import files
files.download(csv_path)


📊 Customer Segmentation Summary:


,Segment,Customer Count
0,Others,1591
1,Champions,1282
2,Hibernating,1260
3,Loyal Customers,703
4,Potential Loyalists,475
5,At Risk,359
6,New Customers,190


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 📊 Step 8: Interactive RFM Dashboard with Plotly

To better understand and explore customer segments, we created a set of **interactive visualizations** using Plotly. These charts allow us to analyze the distribution, value, and behavior of each RFM-defined customer group.

---

### 📦 Dataset Used
The visualizations are based on the `rfm_customer_segments.csv` file, which includes:
- `CustomerID`
- RFM values (`Recency`, `Frequency`, `Monetary`)
- RFM scores (1 to 5)
- Combined segment label (`RFM_Segment`)
- Business segment (e.g. "Champions", "At Risk", etc.)

---

### 📈 What This Code Produces

#### 1. **Bar Chart – Customer Count per Segment**
Displays how many customers fall into each business-defined segment such as:
- Champions
- Loyal Customers
- Hibernating
- At Risk

This helps identify which segments dominate the customer base.

---

#### 2. **Box Plot – Monetary Distribution by Segment**
Visualizes spending behavior across segments. It shows:
- Median spending
- Range of values (min–max)
- Outliers

Use this to pinpoint which segments spend the most (or least).

---

#### 3. **Scatter Plot – Recency vs. Monetary**
Plots customers by **how recently** they purchased and **how much** they spent. Each point is colored by segment and includes hover info:
- Customer ID
- Frequency of purchases

This is useful for spotting high-value but inactive customers or recent low-spenders.

---

#### 4. **Pie Chart – Segment Proportion**
Shows the relative size of each segment. This quickly answers:
> "What percentage of my customers are at risk or champions?"

---

### 🧠 Why This Matters
Interactive dashboards make customer behavior patterns **immediately visible** and **actionable**. This type of analysis enables:
- Targeted marketing campaigns
- Customer retention strategies
- ROI-focused resource allocation

By visualizing the RFM output, businesses can move from insight to strategy with greater clarity.

---

In [ ]:
# Install plotly if needed
!pip install plotly --quiet

# Load libraries
import pandas as pd
import plotly.express as px

# Load the CSV
rfm = pd.read_csv('/mnt/data/rfm_customer_segments.csv')

# 📊 Bar Chart: Number of Customers by Segment
segment_counts = rfm['Segment'].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Customer Count']

fig_segment = px.bar(
    segment_counts,
    x='Segment',
    y='Customer Count',
    color='Segment',
    title='Customer Count per Segment'
)
fig_segment.show()

# 📈 Box Plot: Monetary Distribution by Segment
fig_monetary = px.box(
    rfm,
    x='Segment',
    y='Monetary',
    color='Segment',
    title='Monetary Value Distribution by Segment'
)
fig_monetary.show()

# 📉 Scatter Plot: Recency vs Monetary by Segment
fig_scatter = px.scatter(
    rfm,
    x='Recency',
    y='Monetary',
    color='Segment',
    hover_data=['CustomerID', 'Frequency'],
    title='Recency vs Monetary by Customer Segment'
)
fig_scatter.show()

# 📍 Pie Chart: Segment Proportion
fig_pie = px.pie(
    segment_counts,
    names='Segment',
    values='Customer Count',
    title='Customer Segment Proportions'
)
fig_pie.show()


## 📊 Interpreting the Dashboard & Strategic Recommendations

After performing RFM segmentation and visualizing the customer base, we can now interpret the data to drive targeted business decisions. Below is a breakdown of what we observe from the dashboard and how businesses can act on each insight.

---

### 📌 1. Customer Distribution by Segment (Bar Chart & Pie Chart)

The bar and pie charts show the relative size of each RFM segment:

| Segment              | Customer Count | Approx. % | Insight                                             |
|----------------------|----------------|------------|------------------------------------------------------|
| **Others**            | 1,591          | ~27%       | These customers don't fit a clear behavioral profile |
| **Champions**         | ~1,280         | ~22%       | Highly engaged and high-value customers              |
| **Hibernating**       | ~1,250         | ~22%       | Previously active but now dormant                   |
| **Loyal Customers**   | ~700           | ~12%       | Frequent buyers with solid monetary value            |
| **Potential Loyalists** | ~480         | ~8%        | Recently active, worth nurturing                    |
| **At Risk**           | ~360           | ~6%        | High spenders who have gone silent                   |
| **New Customers**     | ~190           | ~3%        | Just acquired, limited interaction                   |

🧠 **Key Takeaway**: While Champions and Loyal Customers represent strong value, over 30% of the customer base is dormant or undefined, and many segments show potential for reactivation or conversion.

---

### 📈 2. Monetary Distribution by Segment (Box Plot)

- **Champions** have by far the **highest spending**, with extreme outliers above $600,000.
- **Loyal Customers** and **At Risk** groups also show solid monetary value.
- **Hibernating**, **New Customers**, and **Others** spend far less on average.

💡 **Recommendation**:
- **Champions**: Offer VIP perks, loyalty rewards, and early access to products.
- **At Risk**: Re-engagement campaigns with personalized discounts or win-back emails.
- **Others & Hibernating**: Survey or reintroduce brand value with drip campaigns.

---

### 📉 3. Recency vs. Monetary (Scatter Plot)

- High-spending customers cluster on the **low recency end** (i.e., they purchased recently).
- Customers with **older recency values** tend to have low or moderate monetary contributions.
- **Champions** dominate the bottom-left (recent + high spend), while **Others** and **Hibernating** spread across low value and long inactivity.

🎯 **Strategies by Zone**:
- **Bottom-left (Champions)**: Maintain satisfaction, upsell, and refer-a-friend programs.
- **Top-right (Dormant & Low Spend)**: Clean up list, or use low-cost retargeting.
- **Middle Recency + Low Monetary**: Encourage higher engagement (e.g., bundles, subscriptions).

---

### 💼 Strategic Actions by Segment

| Segment              | Strategy                                                                 |
|----------------------|--------------------------------------------------------------------------|
| **Champions**         | Exclusive offers, loyalty perks, referral incentives                     |
| **Loyal Customers**   | Subscription programs, cross-selling, thank-you campaigns                |
| **Potential Loyalists** | Welcome journeys, education content, convert with promotions           |
| **At Risk**           | Reactivation emails, feedback surveys, urgency-driven offers             |
| **Hibernating**       | Reintroduction emails, soft reminders, personalized messages             |
| **New Customers**     | Onboarding sequences, discounts on second purchase, educational content  |
| **Others**            | Monitor behavior or reclassify with more refined logic or features       |

---

### 🧠 Conclusion

The RFM dashboard offers powerful insights into customer behavior. By segmenting and visualizing the data:
- Businesses can **maximize ROI** by targeting the right customers at the right time.
- Retention and engagement strategies become **data-driven and personalized**.
- Marketing efforts can focus on **nurturing value**, **reviving risk**, and **rewarding loyalty**.

This step completes the **analytics-to-action** pipeline of RFM segmentation.



---


**Thank you for exploring this project!**
Feel free to fork, extend, or adapt it to your own business challenges. 💼📊

---

For any futher inquiries you can contact me at gusr.quezada@gmail.com